# 01 - Coleta e leitura dos dados de área queimada (MapBiomas)

Coleta via API REST da área queimada mensal em hectares para o Brasil, as 27 unidades federativas e os 6 biomas entre dezembro de 2024 e novembro de 2026.

Os recortes de Brasil, estados e biomas vêm de endpoints independentes na API, então cada nível territorial é salvo em um CSV separado.

## 1. Bibliotecas e verificação dos meses publicados

In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
import pandas as pd
import requests

base_url = "https://plataforma.monitorfogo.mapbiomas.org/api"
respostas = {}

anos_disp = requests.get(f"{base_url}/statistics/years").json()
respostas["statistics/years"] = anos_disp

meses_disp = {}
for ano in [2024, 2025, 2026]:
    if ano in anos_disp:
        r = requests.get(f"{base_url}/statistics/{ano}/months")
        meses_disp[ano] = r.json()
        respostas[f"statistics/{ano}/months"] = meses_disp[ano]
    else:
        meses_disp[ano] = []

datas = pd.period_range("2024-12", "2026-11", freq="M").astype(str)
linhas_cob = []
for ym in datas:
    ano = int(ym[:4])
    mes = int(ym[5:7])
    linhas_cob.append({"ano_mes": ym, "disponivel": mes in meses_disp[ano]})

cobertura = pd.DataFrame(linhas_cob)
cobertura.to_csv("../dados/mapbiomas_cobertura_mensal.csv", index=False)
print("Meses ainda sem publicação:", list(cobertura[~cobertura["disponivel"]]["ano_mes"]))
display(cobertura.tail())

Meses ainda sem publicação: ['2026-11']


,ano_mes,disponivel
19,2026-07,True
20,2026-08,True
21,2026-09,True
22,2026-10,True
23,2026-11,False


## 2. Catálogo de territórios (Brasil, 27 UFs e 6 biomas)

In [2]:
r_busca = requests.get(f"{base_url}/territories/search/Brasil")
dados_busca = r_busca.json()
respostas["territories/search/Brasil"] = dados_busca

codigo_brasil = None
for item in dados_busca:
    if item["type"] == "country" and item["name"] == "Brasil":
        codigo_brasil = item["code"]

territorios = [{
    "tipo": "brasil",
    "endpoint": "country",
    "codigo": codigo_brasil,
    "nome": "Brasil",
    "agrupamento": "state",
}]

parametros = {"year": 2025, "monthStart": 1, "monthEnd": 12}
for endpoint, nome_tipo in [("state", "estado"), ("biome", "bioma")]:
    url = f"{base_url}/statistics/ranking/country/{codigo_brasil}/{endpoint}/raw"
    resposta = requests.get(url, params=parametros)

    if resposta.status_code == 200:
        catalogo = resposta.json()
        respostas[url] = catalogo
        for item in catalogo:
            territorios.append({
                "tipo": nome_tipo,
                "endpoint": endpoint,
                "codigo": item["code"],
                "nome": item["name"],
                "agrupamento": endpoint,
            })

print("Total de territórios listados:", len(territorios))
display(pd.DataFrame(territorios).head())

Total de territórios listados: 34


,tipo,endpoint,codigo,nome,agrupamento
0,brasil,country,0,Brasil,state
1,estado,state,17,Tocantins,state
2,estado,state,21,Maranhão,state
3,estado,state,51,Mato Grosso,state
4,estado,state,22,Piauí,state


## 3. Coleta das séries mensais (`statistics/time-series`)

In [3]:
meses_publicados = list(cobertura[cobertura["disponivel"]]["ano_mes"])
parametros_serie = {"monthStart": 1, "monthEnd": 12}
tabelas = []

for t in territorios:
    url = f"{base_url}/statistics/time-series/{t['endpoint']}/{t['codigo']}/{t['agrupamento']}"
    resposta = requests.get(url, params=parametros_serie)

    if resposta.status_code == 200:
        dados = resposta.json()
        respostas[url] = dados

        tabela = pd.DataFrame(dados["monthly"])
        tabela["tipo_territorio"] = t["tipo"]
        tabela["codigo_territorio"] = t["codigo"]
        tabela["territorio"] = t["nome"]
        tabelas.append(tabela)

df = pd.concat(tabelas, ignore_index=True)
print("Registros brutos coletados:", len(df))

Registros brutos coletados: 3196


## 4. Padronização e exportação do JSON e dos CSVs

In [4]:
df = df.rename(columns={"year": "ano", "month": "mes", "areaHa": "area_queimada_ha"})
df["ano_mes"] = df["ano"].astype(str) + "-" + df["mes"].astype(str).str.zfill(2)

df = df[df["ano_mes"].isin(meses_publicados)].copy()
df = df[["tipo_territorio", "codigo_territorio", "territorio", "ano", "mes", "ano_mes", "area_queimada_ha"]]
df = df.sort_values(["tipo_territorio", "territorio", "ano_mes"]).reset_index(drop=True)

with open("../dados/mapbiomas_respostas_api.json", "w", encoding="utf-8") as f:
    json.dump(respostas, f, ensure_ascii=False, indent=2)

df[df["tipo_territorio"] == "brasil"].to_csv("../dados/mapbiomas_area_queimada_brasil.csv", index=False)
df[df["tipo_territorio"] == "estado"].to_csv("../dados/mapbiomas_area_queimada_estados.csv", index=False)
df[df["tipo_territorio"] == "bioma"].to_csv("../dados/mapbiomas_area_queimada_biomas.csv", index=False)

display(df.head())

,tipo_territorio,codigo_territorio,territorio,ano,mes,ano_mes,area_queimada_ha
0,bioma,1,Amazônia,2024,12,2024-12,959838.80
1,bioma,1,Amazônia,2025,1,2025-01,629657.39
2,bioma,1,Amazônia,2025,2,2025-02,89022.90
3,bioma,1,Amazônia,2025,3,2025-03,55112.88
4,bioma,1,Amazônia,2025,4,2025-04,30003.98


## 5. Conferência dos CSVs gerados

In [5]:
df_brasil = pd.read_csv("../dados/mapbiomas_area_queimada_brasil.csv")
df_estados = pd.read_csv("../dados/mapbiomas_area_queimada_estados.csv")
df_biomas = pd.read_csv("../dados/mapbiomas_area_queimada_biomas.csv")

print("Brasil:", df_brasil.shape, "| Estados:", df_estados.shape, "| Biomas:", df_biomas.shape)
print("Último mês publicado:", df_brasil["ano_mes"].max())
print("\nValores nulos:")
print(df.isna().sum())

Brasil: (23, 7) | Estados: (621, 7) | Biomas: (138, 7)
Último mês publicado: 2026-10

Valores nulos:
tipo_territorio      0
codigo_territorio    0
territorio           0
ano                  0
mes                  0
ano_mes              0
area_queimada_ha     0
dtype: int64
